# Data auditing and cleaning

Run from top to bottom. Quick mode is a patient subset, not final results. Notebook outputs are separate from production reports.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'features.py').exists()), None)
assert ROOT is not None, 'Open the notebook from the project repository'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.notebook_utils import *
QUICK = True  # False uses the entire eligible dataset; keep consistent across notebooks
OUT = ROOT/'reports'/('notebook_quick' if QUICK else 'notebook_full')
OUT.mkdir(parents=True, exist_ok=True)


Load the original CSV and preserve literal None laboratory categories.

In [ ]:
raw = pd.read_csv(ROOT/'data/raw/diabetic_data.csv', dtype=str, keep_default_na=False)
clean = raw.apply(lambda s:s.str.strip()).replace({'?':np.nan,'':np.nan})
display(clean.head().drop(columns=['encounter_id','patient_nbr']))
print('Shape:',clean.shape,'Patients:',clean.patient_nbr.nunique(),'Duplicates:',clean.duplicated().sum())

Missingness and outcome audit

In [ ]:
audit = pd.DataFrame({'missing':clean.isna().sum(),'percent':clean.isna().mean()*100,'unique':clean.nunique()}).sort_values('percent',ascending=False)
display(audit); audit.query('percent > 0').percent.plot.barh(title='Missing values (%)'); plt.show()
clean.readmitted.value_counts().plot.bar(title='Original outcomes'); plt.show()

Validate encounter identifiers and construct the eligible cohort.

In [ ]:
from src.features import EXCLUDE, NUM
clean=clean.drop_duplicates().copy()
assert clean.encounter_id.notna().all() and clean.patient_nbr.notna().all() and clean.encounter_id.is_unique
assert clean.readmitted.isin(['<30','>30','NO']).all()
for c in NUM[:-1]:
    clean[c]=pd.to_numeric(clean[c],errors='raise'); assert (clean[c]>=0).all()
assert clean.time_in_hospital.between(1,14).all()
clean['target']=clean.readmitted.eq('<30').astype(int)
display(clean.groupby('discharge_disposition_id').target.agg(['size','sum','mean']))
print('Excluded encounters:',clean.discharge_disposition_id.isin(EXCLUDE).sum())
audit.to_csv(OUT/'audit.csv')